In [1]:
import os
import random
import numpy as np
import torch


In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATASET_PATH = r"G:\Ragib\Prescription\Full_Dataset_Split"
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.05, 0.05)),
    transforms.ColorJitter(contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(os.path.join(DATASET_PATH, "Train"), train_transform)
val_ds   = datasets.ImageFolder(os.path.join(DATASET_PATH, "Val"), val_test_transform)
test_ds  = datasets.ImageFolder(os.path.join(DATASET_PATH, "Test"), val_test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

NUM_CLASSES = len(train_ds.classes)
print("Number of classes:", NUM_CLASSES)


Number of classes: 317


In [3]:
import torch.nn as nn
from torchvision import models

class FusionBiLSTMModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # CNN backbones
        self.mobilenet = models.mobilenet_v2(pretrained=True)
        self.mobilenet.classifier = nn.Identity()   # 1280

        self.efficientnet = models.efficientnet_b0(pretrained=True)
        self.efficientnet.classifier = nn.Identity()  # 1280

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=2560,
            hidden_size=256,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        f1 = self.mobilenet(x)
        f2 = self.efficientnet(x)

        fused = torch.cat([f1, f2], dim=1)   # (B, 2560)
        fused = fused.unsqueeze(1)           # (B, 1, 2560)

        lstm_out, _ = self.lstm(fused)
        out = lstm_out[:, -1, :]              # (B, 512)

        return self.classifier(out)


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

_preview_model = FusionBiLSTMModel(NUM_CLASSES).to(DEVICE)
total_params, trainable_params = count_parameters(_preview_model)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\nLayer-wise parameters:")
for name, module in _preview_model.named_children():
    p = sum(x.numel() for x in module.parameters())
    print(f"{name}: {p:,}")

del _preview_model
torch.cuda.empty_cache()


g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` 

Total parameters: 12,215,481
Trainable parameters: 12,215,481

Layer-wise parameters:
mobilenet: 2,223,872
efficientnet: 4,007,548
lstm: 5,771,264
classifier: 212,797


In [5]:
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_epoch(loader):
    model.train()
    total_loss, correct = 0, 0

    for x, y in tqdm(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


def eval_epoch(loader):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


In [6]:
import json
import time
import random
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

# =========================
# MULTI-RUN CONFIGURATION
# =========================
NUM_RUNS = 3
RANDOM_SEEDS = [42, 123, 456]
OUTPUT_DIR = "multirun_bilstm"
EPOCHS = 25

def save_checkpoint(ckpt_dict, checkpoint_path):
    """Write checkpoint atomically to avoid corruption on power loss."""
    tmp_path = checkpoint_path + ".tmp"
    torch.save(ckpt_dict, tmp_path)
    os.replace(tmp_path, checkpoint_path)

def try_load_checkpoint(checkpoint_path, best_model_path, history_path, model, optimizer, device):
    """Load training state from checkpoint, with fallbacks if file is corrupted."""
    start_epoch = 0
    best_val_acc = 0.0
    best_epoch = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    loaded_from_checkpoint = False

    if os.path.exists(checkpoint_path):
        print("Resuming from checkpoint")
        try:
            ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            start_epoch = ckpt["epoch"] + 1
            best_val_acc = ckpt["best_val_acc"]
            best_epoch = ckpt.get("best_epoch", start_epoch)
            train_losses = ckpt["train_losses"]
            val_losses = ckpt["val_losses"]
            train_accs = ckpt["train_accs"]
            val_accs = ckpt["val_accs"]
            loaded_from_checkpoint = True
        except (RuntimeError, EOFError, OSError) as e:
            corrupt_path = checkpoint_path + ".corrupt"
            os.replace(checkpoint_path, corrupt_path)
            print(f"Checkpoint corrupted ({e}). Renamed to {os.path.basename(corrupt_path)}")
    elif os.path.exists(checkpoint_path + ".corrupt"):
        print("Previously corrupted checkpoint detected — using fallback recovery")

    if loaded_from_checkpoint:
        return start_epoch, best_val_acc, best_epoch, train_losses, val_losses, train_accs, val_accs

    if os.path.exists(history_path):
        with open(history_path, "r", encoding="utf-8") as f:
            history = json.load(f)
        train_losses = history.get("train_losses", [])
        val_losses = history.get("val_losses", [])
        train_accs = history.get("train_accs", [])
        val_accs = history.get("val_accs", [])
        best_val_acc = history.get("best_val_acc", max(val_accs) if val_accs else 0.0)
        best_epoch = history.get("best_epoch", 0)
        start_epoch = len(train_losses)
        print(f"Restored training history ({start_epoch} epochs completed)")

    if os.path.exists(best_model_path):
        print("Falling back to best model weights")
        model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=False))
        if start_epoch == 0:
            print("No epoch history found — training continues from best weights at epoch 1")
        else:
            print(f"Resuming from epoch {start_epoch + 1}")
    elif start_epoch == 0:
        print("No recoverable state found — starting from scratch")

    return start_epoch, best_val_acc, best_epoch, train_losses, val_losses, train_accs, val_accs

os.makedirs(OUTPUT_DIR, exist_ok=True)
all_run_results = []

for run_idx in range(NUM_RUNS):
    run_num = run_idx + 1
    seed = RANDOM_SEEDS[run_idx]
    run_dir = os.path.join(OUTPUT_DIR, f"run_{run_num}")
    os.makedirs(run_dir, exist_ok=True)

    complete_flag = os.path.join(run_dir, "run_complete.json")
    if os.path.exists(complete_flag):
        print(f"\n✅ Run {run_num} already complete. Loading saved results.")
        with open(complete_flag, "r", encoding="utf-8") as f:
            all_run_results.append(json.load(f))
        continue

    print(f"\n{'='*60}")
    print(f"Starting Run {run_num}/{NUM_RUNS} | Seed: {seed}")
    print(f"{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

    CHECKPOINT_PATH = os.path.join(run_dir, "checkpoint_bilstm_317.pth")
    BEST_MODEL_PATH = os.path.join(run_dir, "best_bilstm_317.pth")
    HISTORY_PATH = os.path.join(run_dir, "training_history.json")

    model = FusionBiLSTMModel(NUM_CLASSES).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    start_epoch, best_val_acc, best_epoch, train_losses, val_losses, train_accs, val_accs = (
        try_load_checkpoint(
            CHECKPOINT_PATH, BEST_MODEL_PATH, HISTORY_PATH, model, optimizer, DEVICE
        )
    )

    train_start = time.time()

    for epoch in range(start_epoch, EPOCHS):
        tr_loss, tr_acc = train_epoch(train_loader)
        val_loss, val_acc = eval_epoch(val_loader)

        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        train_accs.append(tr_acc)
        val_accs.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), BEST_MODEL_PATH)

        save_checkpoint({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
            "best_epoch": best_epoch,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs
        }, CHECKPOINT_PATH)

        with open(HISTORY_PATH, "w", encoding="utf-8") as f:
            json.dump({
                "train_losses": train_losses,
                "val_losses": val_losses,
                "train_accs": train_accs,
                "val_accs": val_accs,
                "best_val_acc": best_val_acc,
                "best_epoch": best_epoch,
            }, f, indent=2)

        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    training_time = time.time() - train_start

    plt.figure()
    plt.plot(train_accs, label="Train Acc")
    plt.plot(val_accs, label="Val Acc")
    plt.legend(); plt.grid()
    plt.savefig(os.path.join(run_dir, "bilstm_accuracy_curve.png"), dpi=300)
    plt.close()

    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.legend(); plt.grid()
    plt.savefig(os.path.join(run_dir, "bilstm_loss_curve.png"), dpi=300)
    plt.close()

    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model.eval()

    y_true, y_pred = [], []
    y_pred_top5 = []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            probs = torch.softmax(model(x), dim=1)
            preds = probs.argmax(1).cpu().numpy()
            y_true.extend(y.numpy())
            y_pred.extend(preds)
            y_pred_top5.extend(torch.topk(probs, 5, dim=1).indices.cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro"
    )

    print("Precision (macro):", precision)
    print("Recall (macro):", recall)
    print("F1-score (macro):", f1)

    top1_acc = accuracy_score(y_true, y_pred)
    top5_acc = sum(
        y_true[i] in y_pred_top5[i]
        for i in range(len(y_true))
    ) / len(y_true)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12,10))
    sns.heatmap(cm, cmap="Blues", xticklabels=False, yticklabels=False)
    plt.title("BiLSTM Confusion Matrix")
    plt.savefig(os.path.join(run_dir, "confusion_matrix.png"), dpi=300)
    plt.close()

    run_result = {
        "Run Number": run_num,
        "Random Seed": seed,
        "Top-1 Accuracy": top1_acc,
        "Top-5 Accuracy": top5_acc,
        "Precision": p_macro,
        "Recall": r_macro,
        "Macro F1": f1_macro,
        "Weighted F1": f1_weighted,
        "Training Time": training_time,
        "Best Epoch": best_epoch,
    }
    all_run_results.append(run_result)

    with open(complete_flag, "w", encoding="utf-8") as f:
        json.dump(run_result, f, indent=2)

    with open(os.path.join(run_dir, "training_history.json"), "w", encoding="utf-8") as f:
        json.dump({
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs,
        }, f, indent=2)

    del model
    torch.cuda.empty_cache()

print(f"\nCompleted {len(all_run_results)} runs.")



✅ Run 1 already complete. Loading saved results.

✅ Run 2 already complete. Loading saved results.

Starting Run 3/3 | Seed: 456
Previously corrupted checkpoint detected — using fallback recovery
Falling back to best model weights


g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` 

No epoch history found — training continues from best weights at epoch 1


100%|██████████| 971/971 [03:03<00:00,  5.30it/s]


Epoch 1/25
Train Loss: 4.0920 | Train Acc: 0.1715
Val   Loss: 3.5865 | Val   Acc: 0.3669


100%|██████████| 971/971 [01:54<00:00,  8.47it/s]


... (43 outputs left out)

100%|██████████| 971/971 [01:55<00:00,  8.43it/s]


Epoch 24/25
Train Loss: 1.0562 | Train Acc: 0.9983
Val   Loss: 1.0077 | Val   Acc: 0.9883


100%|██████████| 971/971 [01:53<00:00,  8.58it/s]


Epoch 25/25
Train Loss: 1.0444 | Train Acc: 0.9992
Val   Loss: 1.0092 | Val   Acc: 0.9880
Precision (macro): 0.9898518060978628
Recall (macro): 0.9888156008029826
F1-score (macro): 0.9887662604848176

Completed 3 runs.


In [8]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np

metric_cols = [
    "Top-1 Accuracy",
    "Top-5 Accuracy",
    "Precision",
    "Recall",
    "Macro F1",
    "Weighted F1",
    "Training Time",
    "Best Epoch",
]

results_df = pd.DataFrame(all_run_results)
results_df = results_df[[
    "Run Number", "Random Seed",
    "Top-1 Accuracy", "Top-5 Accuracy",
    "Precision", "Recall", "Macro F1", "Weighted F1",
    "Training Time", "Best Epoch",
]]

summary_rows = []
for col in metric_cols:
    mean_val = results_df[col].mean()
    std_val = results_df[col].std(ddof=1) if len(results_df) > 1 else 0.0
    summary_rows.append({
        "Metric": col,
        "Mean ± Standard Deviation": f"{mean_val:.6f} ± {std_val:.6f}",
        "Mean": mean_val,
        "Standard Deviation": std_val,
    })

summary_df = pd.DataFrame(summary_rows)

csv_path = os.path.join(OUTPUT_DIR, "summary_results.csv")
xlsx_path = os.path.join(OUTPUT_DIR, "summary_results.xlsx")

with open(csv_path, "w", encoding="utf-8") as f:
    f.write("Per-Run Results\n")
    results_df.to_csv(f, index=False)
    f.write("\nMean ± Standard Deviation\n")
    summary_df[["Metric", "Mean ± Standard Deviation"]].to_csv(f, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="Per-Run Results", index=False)
    summary_df[["Metric", "Mean ± Standard Deviation"]].to_excel(
        writer, sheet_name="Mean ± Std", index=False
    )

print("Saved:", csv_path)
print("Saved:", xlsx_path)
print("\nPer-Run Results:")
print(results_df.to_string(index=False))
print("\nMean ± Standard Deviation:")
print(summary_df[["Metric", "Mean ± Standard Deviation"]].to_string(index=False))


Saved: multirun_bilstm\summary_results.csv
Saved: multirun_bilstm\summary_results.xlsx

Per-Run Results:
 Run Number  Random Seed  Top-1 Accuracy  Top-5 Accuracy  Precision   Recall  Macro F1  Weighted F1  Training Time  Best Epoch
          1           42        0.991683        0.997993   0.992387 0.991683  0.991573     0.991573    3093.573585          24
          2          123        0.987095        0.997132   0.989083 0.987095  0.986698     0.986698    2997.689296          18
          3          456        0.988816        0.997419   0.989852 0.988816  0.988766     0.988766    3142.374373          20

Mean ± Standard Deviation:
        Metric Mean ± Standard Deviation
Top-1 Accuracy       0.989198 ± 0.002318
Top-5 Accuracy       0.997515 ± 0.000438
     Precision       0.990441 ± 0.001729
        Recall       0.989198 ± 0.002318
      Macro F1       0.989012 ± 0.002447
   Weighted F1       0.989012 ± 0.002447
 Training Time   3077.879085 ± 73.608294
    Best Epoch      20.666667 ±